# NB01 — Data Ingestion & Exploratory Analysis

**Objectives**:
- Download all 20 tickers + benchmarks + VIX/VVIX + FRED macro data via OpenBB/FRED API
- Handle SQ→XYZ ticker transition (cutover 2025-01-21)
- Compute base features: returns, volatility estimators, momentum indicators
- Summary statistics table: mean, std, skewness, kurtosis, Jarque-Bera, ADF per ticker
- Correlation heatmap (full period + rolling 252-day)
- Drawdown analysis: max drawdown, Calmar ratio, time-to-recovery
- Distribution analysis: QQ plots, KDE vs. normal overlay, heavy-tail diagnostics
- Volume regime identification: z-score spikes coinciding with major events
- Sector-grouped volatility boxplots

**Output**: `master_data.parquet`

**Data Quality Protocol** (CLAUDE.md §2.3):
- Forward-fill up to 5 business days, then interpolate remaining
- Use Adj Close exclusively (handles splits & dividends)
- Flag any single-day |return| > 25% as potential corporate-action artifact
- Never fabricate pre-IPO data for short-history tickers (CRWD, DDOG, PLTR)

In [1]:
import sys, os, warnings
warnings.filterwarnings('ignore')
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from src.config import *
from src.data_loader import *
from src.feature_engineering import *
from src.visualization import *
import logging
logging.basicConfig(level=logging.INFO)
print('Imports OK')

Extensions to add: yfinance@1.6.0

Building...
Imports OK


## 1. Data Download
Download all 20 tickers, benchmarks, VIX/VVIX, FRED macro. Handle SQ→XYZ merger.

In [2]:
# Download all stock tickers
raw_data = download_all_tickers()
print(f'Downloaded {len(raw_data)} tickers')

# Merge SQ → XYZ (Block Inc ticker change 2025-01-21)
xyz_merged = merge_sq_xyz()
raw_data['XYZ'] = xyz_merged
print(f'XYZ merged: {len(xyz_merged)} rows')

# Benchmarks
bench_data = download_benchmarks()
print(f'Downloaded {len(bench_data)} benchmarks')

Downloaded 20 tickers


ERROR:yfinance:$SQ: possibly delisted; no timezone found
ERROR:yfinance:
1 Failed download:
ERROR:yfinance:['SQ']: possibly delisted; no timezone found
[Empty] -> No results found. Try adjusting the query parameters.


XYZ merged: 288 rows
Downloaded 13 benchmarks


## 2. Data Quality Checks
Forward-fill ≤5 days, flag NaNs, detect |return| > 25% spikes.

In [3]:
# Build Adj Close panel with quality checks
adj_close, flags = build_adj_close_panel(raw_data)
print(f'Panel: {adj_close.shape[0]} days × {adj_close.shape[1]} tickers')
print(f'Range: {adj_close.index[0].date()} to {adj_close.index[-1].date()}')
print(f'\nMissing values after cleaning: {adj_close.isna().sum().sum()}')

# Short-history tickers
print('\n--- Short-History Tickers ---')
for t, ipo in SHORT_HISTORY_TICKERS.items():
    n = adj_close[t].dropna().shape[0]
    pct = n / adj_close.shape[0] * 100
    print(f'{t}: IPO/DPO {ipo}, {n} observations ({pct:.1f}% of full window)')
    if n < MIN_OBS_FIGARCH:
        print(f'  ⚠ Below {MIN_OBS_FIGARCH} threshold — FIGARCH will be skipped in NB03')

# Data quality flags summary
print('\n--- Quality Flags ---')
for t in TICKERS:
    if t in flags:
        f = flags[t]
        n_ffill = f['ffill_flag'].sum()
        n_spike = f['spike_flag'].sum()
        if n_ffill > 0 or n_spike > 0:
            print(f'{t}: {n_ffill} forward-fills, {n_spike} return spikes >25%')

Panel: 2524 days × 20 tickers
Range: 2016-03-01 to 2026-03-13

Missing values after cleaning: 5112

--- Short-History Tickers ---
CRWD: IPO/DPO 2019-06-12, 1698 observations (67.3% of full window)
DDOG: IPO/DPO 2019-09-19, 1629 observations (64.5% of full window)
PLTR: IPO/DPO 2020-09-30, 1369 observations (54.2% of full window)
  ⚠ Below 1500 threshold — FIGARCH will be skipped in NB03

--- Quality Flags ---
NVDA: 0 forward-fills, 1 return spikes >25%
SNPS: 0 forward-fills, 1 return spikes >25%
META: 0 forward-fills, 2 return spikes >25%
PLTR: 0 forward-fills, 1 return spikes >25%
SAP: 0 forward-fills, 1 return spikes >25%
PANW: 0 forward-fills, 2 return spikes >25%
DDOG: 0 forward-fills, 1 return spikes >25%
ANET: 0 forward-fills, 2 return spikes >25%
AMD: 0 forward-fills, 2 return spikes >25%


## 3. Returns

In [4]:
log_returns = compute_log_returns(adj_close)
simple_returns = compute_simple_returns(adj_close)
print('Log returns:', log_returns.shape)

Log returns: (2524, 20)


## 4. Summary Statistics

In [5]:
# Summary statistics: mean, std, skewness, kurtosis, Jarque-Bera, ADF
stats_table = summary_statistics(log_returns[[t for t in TICKERS if t in log_returns.columns]])
print(f'Computed summary statistics for {len(stats_table)} tickers')
print(f'\nKey findings:')
print(f'  Highest annualized vol: {stats_table["annualized_vol"].idxmax()} '
      f'({stats_table["annualized_vol"].max():.1%})')
print(f'  Lowest annualized vol:  {stats_table["annualized_vol"].idxmin()} '
      f'({stats_table["annualized_vol"].min():.1%})')
print(f'  Most negatively skewed: {stats_table["skewness"].idxmin()} '
      f'({stats_table["skewness"].min():.3f})')
print(f'  Highest excess kurtosis: {stats_table["excess_kurtosis"].idxmax()} '
      f'({stats_table["excess_kurtosis"].max():.2f})')
print(f'  Jarque-Bera rejects normality (p<0.05): '
      f'{(stats_table["jarque_bera_p"] < 0.05).sum()}/20 tickers')
stats_table.style.format({
    'mean_daily': '{:.6f}', 'std_daily': '{:.4f}',
    'annualized_return': '{:.2%}', 'annualized_vol': '{:.2%}',
    'skewness': '{:.3f}', 'excess_kurtosis': '{:.2f}',
    'jarque_bera_p': '{:.2e}', 'adf_pvalue': '{:.4f}'
})

Computed summary statistics for 20 tickers

Key findings:
  Highest annualized vol: PLTR (69.0%)
  Lowest annualized vol:  MSFT (26.9%)
  Most negatively skewed: SNPS (-3.235)
  Highest excess kurtosis: SNPS (66.05)
  Jarque-Bera rejects normality (p<0.05): 20/20 tickers


,mean_daily,std_daily,annualized_return,annualized_vol,skewness,excess_kurtosis,jarque_bera_stat,jarque_bera_p,adf_stat,adf_pvalue
ticker,,,,,,,,,,
NVDA,0.002138,0.0311,53.88%,49.39%,0.098,6.74,4756.865328,0.00e+00,-18.017286,0.0000
AVGO,0.001247,0.0244,31.44%,38.80%,-0.176,11.39,13587.577974,0.00e+00,-19.358775,0.0000
TSM,0.001039,0.0212,26.19%,33.65%,0.000,4.18,1830.419663,0.00e+00,-10.317893,0.0000
SNPS,0.000871,0.0223,21.94%,35.36%,-3.235,66.05,461187.868395,0.00e+00,-17.143190,0.0000
MSFT,0.000800,0.0169,20.16%,26.86%,-0.262,8.29,7212.469971,0.00e+00,-17.096851,0.0000
AMZN,0.000781,0.0204,19.68%,32.36%,-0.013,5.21,2839.654656,0.00e+00,-51.271016,0.0000
META,0.000682,0.0243,17.19%,38.58%,-1.194,25.88,70738.956978,0.00e+00,-52.383190,0.0000
GOOG,0.000843,0.0180,21.24%,28.60%,-0.181,4.72,2347.412442,0.00e+00,-16.883860,0.0000
AAPL,0.000911,0.0182,22.95%,28.86%,-0.079,6.80,4838.733102,0.00e+00,-16.300574,0.0000


## 5. Stationarity Tests (ADF + KPSS)

In [6]:
stat_tests = stationarity_table(log_returns[[t for t in TICKERS if t in log_returns.columns]])
stat_tests

/Users/laurentnguyen/Documents/Personal Project/Finance/Portfolio Construction/src/feature_engineering.py:501: InterpolationWarning: The test statistic is outside of the range of p-values available in the
look-up table. The actual p-value is greater than the p-value returned.

  stat, p_value, lags, crit = _kpss(clean, regression=regression, nlags="auto")
/Users/laurentnguyen/Documents/Personal Project/Finance/Portfolio Construction/src/feature_engineering.py:501: InterpolationWarning: The test statistic is outside of the range of p-values available in the
look-up table. The actual p-value is greater than the p-value returned.

  stat, p_value, lags, crit = _kpss(clean, regression=regression, nlags="auto")
/Users/laurentnguyen/Documents/Personal Project/Finance/Portfolio Construction/src/feature_engineering.py:501: InterpolationWarning: The test statistic is outside of the range of p-values available in the
look-up table. The actual p-value is greater than the p-value returned.

  stat

,ticker,adf_stat,adf_pvalue,kpss_stat,kpss_pvalue,stationary
0,NVDA,-18.017286,2.702415e-30,0.088702,0.100000,True
1,AVGO,-19.358775,0.000000e+00,0.129800,0.100000,True
2,TSM,-10.317893,3.063010e-18,0.103464,0.100000,True
3,SNPS,-17.143190,7.022406e-30,0.175687,0.100000,True
4,MSFT,-17.096851,7.524069e-30,0.222242,0.100000,True
5,AMZN,-51.271016,0.000000e+00,0.176699,0.100000,True
6,META,-52.383190,0.000000e+00,0.071952,0.100000,True
7,GOOG,-16.883860,1.057798e-29,0.075321,0.100000,True
8,AAPL,-16.300574,3.269404e-29,0.088613,0.100000,True
9,CRM,-17.413585,4.870322e-30,0.193870,0.100000,True


## 5b. BH-FDR Correction on Stationarity Tests

Apply Benjamini-Hochberg False Discovery Rate correction (q=0.05) to
ADF and KPSS p-values across all 20 tickers, controlling the expected
proportion of false discoveries when testing the same hypothesis multiple times.

In [7]:
from src.statistical_tests import benjamini_hochberg

# Apply BH-FDR to ADF p-values across 20 tickers
adf_pvals = stat_tests['adf_pvalue'].values
rejected_adf, adjusted_adf = benjamini_hochberg(adf_pvals, q=0.05)
stat_tests['adf_pvalue_bh'] = adjusted_adf
stat_tests['adf_reject_bh'] = rejected_adf

# Apply BH-FDR to KPSS p-values
kpss_pvals = stat_tests['kpss_pvalue'].values
rejected_kpss, adjusted_kpss = benjamini_hochberg(kpss_pvals, q=0.05)
stat_tests['kpss_pvalue_bh'] = adjusted_kpss
stat_tests['kpss_reject_bh'] = rejected_kpss

print("--- BH-FDR Corrected Stationarity Tests ---")
print(f"ADF: {sum(rejected_adf)}/{len(rejected_adf)} reject H0 (unit root) after BH correction")
print(f"KPSS: {sum(rejected_kpss)}/{len(rejected_kpss)} reject H0 (stationarity) after BH correction")
print("\nConfirmatory approach: ADF rejects unit root + KPSS fails to reject stationarity → stationary")
stat_tests[['ticker', 'adf_pvalue', 'adf_pvalue_bh', 'adf_reject_bh',
            'kpss_pvalue', 'kpss_pvalue_bh', 'kpss_reject_bh']]

--- BH-FDR Corrected Stationarity Tests ---
ADF: 20/20 reject H0 (unit root) after BH correction
KPSS: 0/20 reject H0 (stationarity) after BH correction

Confirmatory approach: ADF rejects unit root + KPSS fails to reject stationarity → stationary


,ticker,adf_pvalue,adf_pvalue_bh,adf_reject_bh,kpss_pvalue,kpss_pvalue_bh,kpss_reject_bh
0,NVDA,2.702415e-30,4.959729e-30,True,0.100000,0.1,False
1,AVGO,0.000000e+00,0.000000e+00,True,0.100000,0.1,False
2,TSM,3.063010e-18,3.063010e-18,True,0.100000,0.1,False
3,SNPS,7.022406e-30,1.074867e-29,True,0.100000,0.1,False
4,MSFT,7.524069e-30,1.074867e-29,True,0.100000,0.1,False
5,AMZN,0.000000e+00,0.000000e+00,True,0.100000,0.1,False
6,META,0.000000e+00,0.000000e+00,True,0.100000,0.1,False
7,GOOG,1.057798e-29,1.410397e-29,True,0.100000,0.1,False
8,AAPL,3.269404e-29,4.086754e-29,True,0.100000,0.1,False
9,CRM,4.870322e-30,8.117204e-30,True,0.100000,0.1,False


## 6. Cumulative Returns

In [8]:
fig = plot_cumulative_returns(adj_close[[t for t in TICKERS if t in adj_close.columns]], save_name='nb01_cumulative_returns')
plt.show()

INFO:src.visualization:Saved figure: /Users/laurentnguyen/Documents/Personal Project/Finance/Portfolio Construction/outputs/figures/nb01_cumulative_returns.png


## 7. Correlation Heatmap

In [9]:
corr = log_returns[[t for t in TICKERS if t in log_returns.columns]].dropna().corr()
fig = plot_correlation_heatmap(corr, title='Full-Period Correlation Matrix (Log Returns)',
                                save_name='nb01_correlation_full')
plt.show()

INFO:src.visualization:Saved figure: /Users/laurentnguyen/Documents/Personal Project/Finance/Portfolio Construction/outputs/figures/nb01_correlation_full.png


## 7b. Rolling 252-Day Correlation (Selected Pairs)

Track how correlations evolve over time — especially through regime shifts
(COVID crash, rate hikes, AI boom). Correlation instability is a key risk factor.

In [10]:
# Rolling 252-day pairwise correlations for key pairs
key_pairs = [
    ('NVDA', 'AMD'),    # Semiconductor competitors
    ('NVDA', 'TSM'),    # GPU designer vs foundry
    ('MSFT', 'AAPL'),   # Mega-cap anchors
    ('CRWD', 'PANW'),   # Cybersecurity peers
    ('META', 'GOOG'),   # Ad-tech competitors
    ('NVDA', 'MSFT'),   # Cross-sector: semi vs cloud
]

fig, axes = plt.subplots(3, 2, figsize=(16, 12), sharex=True)
for ax, (a, b) in zip(axes.flatten(), key_pairs):
    roll_corr = rolling_correlation(log_returns[a], log_returns[b], window=252)
    ax.plot(roll_corr.index, roll_corr.values, linewidth=0.8)
    ax.axhline(y=roll_corr.mean(), color='red', linestyle='--', alpha=0.5,
               label=f'Mean: {roll_corr.mean():.2f}')
    ax.set_title(f'{a} vs {b}', fontsize=11)
    ax.set_ylabel('Correlation')
    ax.legend(fontsize=8)
    ax.set_ylim(-0.2, 1.0)
    # Shade COVID crash
    ax.axvspan('2020-02-19', '2020-03-23', alpha=0.15, color='red')
    # Shade rate hike cycle
    ax.axvspan('2022-01-03', '2022-10-13', alpha=0.1, color='orange')

fig.suptitle('Rolling 252-Day Pairwise Correlations', fontsize=14, y=1.01)
fig.tight_layout()
save_fig(fig, 'nb01_rolling_correlations')
plt.show()

INFO:src.visualization:Saved figure: /Users/laurentnguyen/Documents/Personal Project/Finance/Portfolio Construction/outputs/figures/nb01_rolling_correlations.png


## 8. Drawdown Analysis

## 9. Distribution Diagnostics

QQ plots reveal departure from normality (heavy tails, skewness).
KDE overlays show the empirical density vs. the normal assumption.
All 20 tickers reject normality via Jarque-Bera (expected for daily returns).

In [11]:
# QQ plots for all 20 tickers (4×5 grid)
fig, axes = plt.subplots(4, 5, figsize=(20, 16))
from scipy import stats as sp_stats
for ax, t in zip(axes.flatten(), TICKERS):
    data = log_returns[t].dropna()
    sp_stats.probplot(data, dist="norm", plot=ax)
    ax.set_title(t, fontsize=10)
    ax.get_lines()[0].set_markersize(1.5)
fig.suptitle('QQ Plots vs Normal Distribution — All 20 Tickers', fontsize=14, y=1.01)
fig.tight_layout()
save_fig(fig, 'nb01_qq_all_tickers')
plt.show()

INFO:src.visualization:Saved figure: /Users/laurentnguyen/Documents/Personal Project/Finance/Portfolio Construction/outputs/figures/nb01_qq_all_tickers.png


## 10. Volume Spike Detection & Event Mapping

Volume z-score spikes (> 3σ) identify periods of extreme market activity.
We cross-reference these with known events to validate the event study
dates used in NB02.

In [12]:
# Comprehensive volume spike detection across all 20 tickers
all_vol_spikes = []
for t in TICKERS:
    if t not in raw_data or raw_data[t].empty:
        continue
    vol_col = 'Volume' if 'Volume' in raw_data[t].columns else None
    if vol_col is None:
        continue
    vz = volume_zscore(raw_data[t][vol_col], 20)
    spikes = vz[vz > 3].dropna()
    for date, zscore in spikes.items():
        # Map to nearest key event
        event_match = 'Unknown'
        for event_name, (ev_start, ev_end) in KEY_EVENTS.items():
            if pd.Timestamp(ev_start) - pd.Timedelta(days=5) <= date <= pd.Timestamp(ev_end) + pd.Timedelta(days=5):
                event_match = event_name
                break
        all_vol_spikes.append({
            'ticker': t, 'date': date, 'volume_zscore': zscore,
            'event': event_match
        })

spike_df = pd.DataFrame(all_vol_spikes)
print(f'Total volume spikes (>3σ): {len(spike_df)} across {spike_df["ticker"].nunique()} tickers')

# Event attribution summary
print('\n--- Spikes by Event ---')
event_counts = spike_df.groupby('event').size().sort_values(ascending=False)
for event, count in event_counts.items():
    affected_tickers = spike_df[spike_df['event'] == event]['ticker'].unique()
    print(f'  {event:30s}: {count:3d} spikes across {len(affected_tickers)} tickers')

# Visualize volume spikes timeline for top-4 highest-spike tickers
top_spike_tickers = spike_df['ticker'].value_counts().head(4).index.tolist()
fig, axes = plt.subplots(len(top_spike_tickers), 1, figsize=(16, 3*len(top_spike_tickers)), sharex=True)
for ax, t in zip(axes, top_spike_tickers):
    if t not in raw_data or raw_data[t].empty:
        continue
    vz = volume_zscore(raw_data[t]['Volume'], 20)
    ax.plot(vz.index, vz.values, linewidth=0.5, alpha=0.7)
    ax.axhline(y=3, color='red', linestyle='--', alpha=0.5, label='3σ threshold')
    # Mark key events
    for event_name, (ev_start, ev_end) in KEY_EVENTS.items():
        ax.axvspan(ev_start, ev_end, alpha=0.15, color='orange')
    ax.set_ylabel(f'{t} Vol Z-Score')
    ax.set_ylim(-3, 15)
axes[0].legend()
fig.suptitle('Volume Z-Score with Key Event Windows (orange)', fontsize=13)
fig.tight_layout()
save_fig(fig, 'nb01_volume_spikes')
plt.show()

Total volume spikes (>3σ): 1099 across 20 tickers

--- Spikes by Event ---
  Unknown                       : 970 spikes across 20 tickers
  Inflation / Rate Hikes        :  84 spikes across 19 tickers
  DeepSeek / Tariff Shock       :  14 spikes across 14 tickers
  COVID Crash                   :  12 spikes across 8 tickers
  SVB Collapse                  :  12 spikes across 7 tickers
  CrowdStrike Outage            :   3 spikes across 3 tickers
  ChatGPT / AI Rally            :   2 spikes across 2 tickers
  Chip Export Ban v2            :   2 spikes across 2 tickers


INFO:src.visualization:Saved figure: /Users/laurentnguyen/Documents/Personal Project/Finance/Portfolio Construction/outputs/figures/nb01_volume_spikes.png


In [13]:
dd_stats = []
for t in TICKERS:
    p = adj_close[t].dropna()
    if len(p) < 2: continue
    dd_stats.append({'ticker': t, 'max_drawdown': max_drawdown(p), 'calmar': calmar_ratio(p)})
dd_df = pd.DataFrame(dd_stats).set_index('ticker').sort_values('max_drawdown')
dd_df

,max_drawdown,calmar
ticker,,
PLTR,-0.846154,0.784465
META,-0.767361,0.244259
DDOG,-0.681064,0.299176
CRWD,-0.676922,0.519485
NVDA,-0.663621,1.075264
AMD,-0.654499,0.863116
CRM,-0.586172,0.181575
MU,-0.578234,0.760512
TSM,-0.571449,0.523669


In [14]:
# Build and save master data (combines all tickers + benchmarks + macro)
master = load_or_build_master(force_rebuild=True)
print(f'Master data: {master.shape[0]} rows × {master.shape[1]} columns')
print(f'Saved to: {MASTER_DATA_FILE}')

# Also save summary statistics and stationarity tests as CSV for reference
stats_table.to_csv(TABLES_DIR / 'nb01_summary_statistics.csv')
stat_tests.to_csv(TABLES_DIR / 'nb01_stationarity_tests.csv', index=False)
dd_df.to_csv(TABLES_DIR / 'nb01_drawdown_stats.csv')
print(f'\nSaved supplementary tables to {TABLES_DIR}')

print('\n--- NB01 Complete ---')
print(f'  Tickers processed: {len(TICKERS)}')
print(f'  Date range: {master.index[0].date()} to {master.index[-1].date()}')
print(f'  Trading days: {master.shape[0]}')
print(f'  Columns: {master.shape[1]} (tickers + benchmarks + macro)')
print(f'\nReady for downstream notebooks: NB02, NB03, NB05, NB06 (parallelizable)')

INFO:src.data_loader:Saved master_data.parquet  (2526 rows × 33 cols)


Master data: 2526 rows × 33 columns
Saved to: /Users/laurentnguyen/Documents/Personal Project/Finance/Portfolio Construction/data/processed/master_data.parquet

Saved supplementary tables to /Users/laurentnguyen/Documents/Personal Project/Finance/Portfolio Construction/outputs/tables

--- NB01 Complete ---
  Tickers processed: 20
  Date range: 2016-03-01 to 2026-03-13
  Trading days: 2526
  Columns: 33 (tickers + benchmarks + macro)

Ready for downstream notebooks: NB02, NB03, NB05, NB06 (parallelizable)


## 9. Distribution Diagnostics

In [15]:
for t in TICKERS[:5]:
    fig = plot_qq(log_returns[t], title=f'QQ: {t}', save_name=f'nb01_qq_{t}')
    plt.show()

INFO:src.visualization:Saved figure: /Users/laurentnguyen/Documents/Personal Project/Finance/Portfolio Construction/outputs/figures/nb01_qq_NVDA.png
INFO:src.visualization:Saved figure: /Users/laurentnguyen/Documents/Personal Project/Finance/Portfolio Construction/outputs/figures/nb01_qq_AVGO.png
INFO:src.visualization:Saved figure: /Users/laurentnguyen/Documents/Personal Project/Finance/Portfolio Construction/outputs/figures/nb01_qq_TSM.png
INFO:src.visualization:Saved figure: /Users/laurentnguyen/Documents/Personal Project/Finance/Portfolio Construction/outputs/figures/nb01_qq_SNPS.png
INFO:src.visualization:Saved figure: /Users/laurentnguyen/Documents/Personal Project/Finance/Portfolio Construction/outputs/figures/nb01_qq_MSFT.png


## 10. Volume Spike Detection

## 10b. ADR Premium Analysis (TSM, SAP)

Track ADR premium/discount for the two non-US tickers:
- TSM premium = (P_ADR / 5) / (P_{2330.TW} · FX_{TWD/USD}) - 1
- SAP premium = P_ADR / (P_{SAP.DE} · FX_{EUR/USD}) - 1

Persistent non-zero ADR premium indicates capital flow restrictions or liquidity preference.

In [16]:
# ADR Premium Analysis for TSM and SAP (optional enrichment)
try:
    from src.data_loader import download_single_ticker

    tsm_local = download_single_ticker('2330.TW', START_DATE, END_DATE)
    sap_local = download_single_ticker('SAP.DE', START_DATE, END_DATE)
    twdusd = download_single_ticker('TWDUSD=X', START_DATE, END_DATE)
    eurusd = download_single_ticker('EURUSD=X', START_DATE, END_DATE)

    # TSM ADR ratio = 5:1 (1 ADR = 5 ordinary shares)
    tsm_premium = (adj_close['TSM'] / 5) / (tsm_local['Adj Close'] * twdusd['Adj Close']) - 1
    sap_premium = adj_close['SAP'] / (sap_local['Adj Close'] * eurusd['Adj Close']) - 1

    fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=True)
    axes[0].plot(tsm_premium.dropna(), linewidth=0.7)
    axes[0].axhline(0, color='red', linestyle='--', alpha=0.5)
    axes[0].set_title(f'TSM ADR Premium (mean: {tsm_premium.mean():.2%})')
    axes[0].set_ylabel('Premium/Discount')

    axes[1].plot(sap_premium.dropna(), linewidth=0.7)
    axes[1].axhline(0, color='red', linestyle='--', alpha=0.5)
    axes[1].set_title(f'SAP ADR Premium (mean: {sap_premium.mean():.2%})')
    axes[1].set_ylabel('Premium/Discount')

    fig.suptitle('ADR Premium/Discount Analysis', fontsize=13)
    fig.tight_layout()
    save_fig(fig, 'nb01_adr_premium')
    plt.show()
except Exception as e:
    print(f'ADR premium analysis skipped: {e}')

INFO:src.visualization:Saved figure: /Users/laurentnguyen/Documents/Personal Project/Finance/Portfolio Construction/outputs/figures/nb01_adr_premium.png


In [17]:
for t in ['NVDA', 'CRWD', 'META', 'TSM']:
    if t not in raw_data or raw_data[t].empty: continue
    vz = volume_zscore(raw_data[t]['Volume'], 20)
    spikes = vz[vz > 3].dropna()
    print(f'{t}: {len(spikes)} volume spikes > 3σ')

NVDA: 54 volume spikes > 3σ
CRWD: 51 volume spikes > 3σ
META: 82 volume spikes > 3σ
TSM: 49 volume spikes > 3σ


## 11. Save Master Data

In [18]:
master = load_or_build_master(force_rebuild=True)
print(f'Saved: {master.shape}, file: {MASTER_DATA_FILE}')

INFO:src.data_loader:Saved master_data.parquet  (2526 rows × 33 cols)


Saved: (2526, 33), file: /Users/laurentnguyen/Documents/Personal Project/Finance/Portfolio Construction/data/processed/master_data.parquet
